# 프로젝트: KoChatGPT 업그레이드 하기


In [2]:
import os
import sys
import random
from copy import deepcopy
import json

import torch
import torch.nn as nn
import evaluate
from datasets import load_dataset
import pandas as pd
import re

from peft import get_peft_model, LoraConfig, TaskType, PeftModel

from transformers import (
    PreTrainedTokenizerFast,
    AutoModelForCausalLM, 
    TrainingArguments, 
    Trainer,
    DataCollatorForLanguageModeling
)


print(f"PyTorch Version: {torch.__version__}")

PyTorch Version: 2.10.0


In [ ]:
CFG = {
    "MODEL_NAME": "skt/ko-gpt-trinity-1.2B-v0.5",
    "KOCHATGPT_PATH": "/Users/jamesyang/Projects/KoChatGPT/data_kochatgpt",
    "DATA_PATH": "data",
    "MODEL_OUTPUT_PATH": "models/SFT",
    "MERGED_SFT_PATH": "models/SFT_merged"

    # 학습 도중 체크포인트들이 저장될 위치
    "TRAIN_OUTPUT_PATH": "kochatgpt-trinity-sft",
    "PROMPT_TEMPLATE": "### Instruction(명령어):\n{prompt}\n\n### Response(응답):",
    "LORA_RANK": 8,
    "MODEL_MAX_LEN": 512,

    "RM_MODEL_OUTPUT_PATH": "models/RM",
    "PPO_MODEL_OUTPUT_PATH": "models/PPO",

    "IS_SFT_TRAINING": True,

    "DEVICE": torch.device("cuda" if torch.cuda.is_available() 
                      else "mps" if torch.backends.mps.is_available() 
                      else "cpu")

}


In [4]:
HOME = os.path.expanduser("~")
BASE_PATH = os.path.join(HOME, "Projects/content")
GPT_PATH =  f"{BASE_PATH}/chatgpt"


if BASE_PATH not in sys.path:
    sys.path.append(BASE_PATH)

from chatgpt.models.base import RewardModel
from chatgpt.trainer.strategies import NaiveStrategy
from chatgpt.trainer.rm import RewardModelTrainer
from chatgpt.dataset import RewardDataset
from chatgpt.models.gpt import GPTActor, GPTCritic
from chatgpt.trainer import PPOTrainer


# [Phase 0] SFT EDA 분석 
- SFT Data에 불필요한 ' 따옴표 예) "'저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다"
- 답변 맨처음에 불필요한 문자 제거 예) "\\n나는"
- '확인이 어렵습니다' 등 회피형 무잭임한 키워드와 무성의(20자미만) 1299개 (무성의/회피형)


In [4]:
import json
import os
import pandas as pd
import re

# 1. 회피형/무책임 키워드 정의 (함수 밖에서 전역으로 관리)
irresponsible_keywords = [
    '알 수 없습니다', '답변을 드릴 수 없습니다', '정보를 찾을 수 없습니다', 
    '조언을 드릴 수 없습니다', '제공하지 않습니다', '문의해 보시기 바랍니다',
    '도움을 드릴 수 없습니다', '확인이 어렵습니다'
]

def clean_text(text):
    """문장 앞뒤 노이즈 완벽 제거 세척기"""
    if not text: return ""
    # 앞뒤 공백, 따옴표, 줄바꿈(\n), 특수기호 제거 패턴
    noise_pattern = r'^[\s\'\"?.!:\n\t>\\n]+|[\s\'\"?.!:\n\t>\\n]+$'
    text = re.sub(noise_pattern, '', text).strip()
    
    # 리터럴 형태의 \\n, \\t 재차 확인 제거
    while text.startswith('\\n') or text.startswith('\\t'): text = text[2:].strip()
    while text.endswith('\\n') or text.endswith('\\t'): text = text[:-2].strip()
    return text



def clean_sft_data(input_path):
    # 경로 및 환경 설정
    os.makedirs(CFG['DATA_PATH'], exist_ok=True)
    out_cleaned = f"{CFG['DATA_PATH']}/kochatgpt_1_SFT_cleaned.jsonl"
    out_bad = f"{CFG['DATA_PATH']}/sft_bad_samples.csv"
    
    with open(input_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    cleaned_data, bad_samples = [], []
    # 키워드 검색용 정규식 최적화
    irresponsible_regex = re.compile('|'.join(irresponsible_keywords))

    for item in data:
        # STEP 1: 세척
        item['completion'] = clean_text(item['completion'])
        comp = item['completion']
        
        # STEP 2: 필터링 (키워드 매칭 OR 20자 미만)
        if irresponsible_regex.search(comp) or len(comp) < 20:
            bad_samples.append(item)
        else:
            cleaned_data.append(item)

    # STEP 3: 저장
    with open(out_cleaned, 'w', encoding='utf-8') as f:
        json.dump(cleaned_data, f, ensure_ascii=False, indent=4)
    pd.DataFrame(bad_samples).to_csv(out_bad, index=False, encoding='utf-8-sig')

    return len(data), len(cleaned_data), len(bad_samples), out_cleaned



# --- 실행 ---
data_path = f"{CFG['KOCHATGPT_PATH']}/kochatgpt_1_SFT.jsonl"
total, clear, bad, save_path = clean_sft_data(data_path)

print(f"✅ 정제 완료: {total} -> {clear} (불량 {bad}개 제외)")


✅ 정제 완료: 12000 -> 10658 (불량 1342개 제외)


### 📈 데이터 증강(Augmentation) 전략

정제 과정에서 제거된 무성의한 데이터를 보충하고, 모델의 대응 능력을 키우기 위해 아래 전략을 적용했습니다.

#### 1. 단어 교체 (Entity Swapping)
- **지명/음식 교체**: 문장 내의 핵심 명사를 유사한 단어로 치환하여 데이터 다양성 확보
- *예: "서울 맛집" → "부산 맛집"*

#### 2. 질문 어투 변형 (Prompt Variation)
- **다양한 종결어미**: "알려줘", "알려주세요", "알고 있나요?" 등 질문 스타일 다변화
- *예: "경복궁 위치" → "경복궁 위치 혹시 알고 있나요?"*

#### 3. 데이터 구성 결과
- **정제 데이터**: 10,754개 (고품질 샘플)
- **증강 데이터**: 1,246개 (변형 생성)
- **최종 데이터**: **12,000개** (정규 사이즈 유지)


In [5]:
# 경로 설정
cleaned_path = f"{CFG['DATA_PATH']}/kochatgpt_1_SFT_cleaned.jsonl"
augmented_path = f"{CFG['DATA_PATH']}/kochatgpt_1_SFT_augmented.jsonl"

def augment_sft_data(input_path, target_count=12000):
    with open(input_path, 'r', encoding='utf-8') as f:
        ds = json.load(f)
    
    current_count = len(ds)
    needed_count = target_count - current_count
    
    print(f"현재 데이터: {current_count}개, 필요한 증강 수: {needed_count}개")
    
    augmented_samples = []
    
    # 교체용 단어 리스트 (Entity Swapping용)
    cities = ['서울', '부산', '대구', '인천', '광주', '대전', '울산', '제주도']
    foods = ['불고기', '비빔밥', '김치찌개', '된장찌개', '삼겹살', '치킨', '떡볶이']
    
    # 1. 증강 시작 (모자란 개수만큼 원본에서 무작위로 뽑아 변형)
    samples_to_augment = random.sample(ds, min(needed_count * 2, current_count))
    
    for item in samples_to_augment:
        if len(augmented_samples) >= needed_count:
            break
            
        new_prompt = item['prompt']
        new_completion = item['completion']
        
        # 기법 A: 질문 어투 변형 (Prompt Variation)
        variation_templates = [
            lambda p: f"{p}에 대해 자세히 알려주세요.",
            lambda p: f"{p} 혹시 알고 있나요?",
            lambda p: f"{p} 설명 부탁드립니다.",
            lambda p: f"혹시 {p}"
        ]
        
        # 기법 B: 단어 교체 (Entity Swapping)
        # 지명 교체
        for city in cities:
            if city in new_prompt:
                target_city = random.choice([c for c in cities if c != city])
                new_prompt = new_prompt.replace(city, target_city)
                new_completion = new_completion.replace(city, target_city)
                break
        
        # 음식 교체
        for food in foods:
            if food in new_prompt:
                target_food = random.choice([f for f in foods if f != food])
                new_prompt = new_prompt.replace(food, target_food)
                new_completion = new_completion.replace(food, target_food)
                break

        # 랜덤하게 질문 어투 하나 적용
        new_prompt = random.choice(variation_templates)(new_prompt)
        
        augmented_samples.append({
            "prompt": new_prompt,
            "completion": new_completion,
            "tokens": item.get("tokens", 0) # 토큰 수는 대략 유지
        })

    # 2. 원본과 합치기
    final_ds = ds + augmented_samples
    
    # 3. 결과 저장
    with open(augmented_path, 'w', encoding='utf-8') as f:
        json.dump(final_ds, f, ensure_ascii=False, indent=4)
        
    return len(final_ds)

final_total = augment_sft_data(cleaned_path)

print(f"✅ 증강 완료!")
print(f"📈 최종 데이터 수: {final_total}개")
print(f"📁 최종 파일 저장: {augmented_path}")


현재 데이터: 10658개, 필요한 증강 수: 1342개
✅ 증강 완료!
📈 최종 데이터 수: 12000개
📁 최종 파일 저장: data/kochatgpt_1_SFT_augmented.jsonl


# [Phase 1] Foundation Model 교체 및 LoRA 설정 (SFT 모델 로드)

- 1단계: 원본 모델 가중치 (12억 개 파라미터) ➔ Freeze
  - Trinity 1.2B 모델 본체의 모든 파라미터는 얼음(Frozen) 상태가 됩니다.
  - **미분(Gradient)**을 계산하지 않고, 값을 바꾸지도 않습니다. 학습 시 메모리를 아주 적게 차지하게 됩니다.
- 2단계: LoRA용 얇은 레이어 (약 1,100만 개 파라미터) ➔ Unfreeze (학습 가능)
  - 원본 모델 옆에 아주 얇은 '어댑터(Adapter)' 레이어를 새로 끼워 넣습니다. 이 부분만 학습 가능하도록 열어둡니다.
  - 우리가 준비한  SFT_dataset 의 지식은 오직 이 아주 작은 레이어에만 기록됩니다.
- 3단계: 합치기 (Inference)
  - 생성할 때는 얼어있는 본체와 학습된 어댑터가 힘을 합쳐 답변을 내놓습니다.

In [5]:
# [셀 3] Foundation Model 로드 및 PEFT(LoRA) 적용

# 토크나이저 로드 (special tokens 추가)
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    CFG["MODEL_NAME"], 
    bos_token='</s>', 
    eos_token='</s>', 
    unk_token='<unk>', 
    pad_token='<pad>', 
    mask_token='<mask>',
    padding_side="left",
    model_max_length=CFG["MODEL_MAX_LEN"]       # 모델이 한 번에 처리할 최대 길이 설정
)

# 모델 로드 (Mac의 경우 device_map="auto" 로 MPS에 적절히 할당되거나 GPU 메모리 최적화를 위해 fp16 사용)
model = AutoModelForCausalLM.from_pretrained(
    CFG["MODEL_NAME"],
    dtype=torch.bfloat16,  # 메모리 절약을 위한 Half Precision
    device_map="auto"
)

# LoRA(Low-Rank Adaptation) 설정
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=CFG["LORA_RANK"],               # Rank 크기 (줄일수록 학습 파라미터가 적어짐)
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["c_attn"] # GPT-2 계열의 Attention 레이어 Linear Projection 타겟
)

# 모델에 LoRA 어댑터 부착
model = get_peft_model(model, peft_config)

# 학습할 파라미터 수 확인 (전체 파라미터 대비 1% 미만으로 OOM 방지)
model.print_trainable_parameters()


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: skt/ko-gpt-trinity-1.2B-v0.5
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable params: 1,474,560 || all params: 1,164,030,720 || trainable%: 0.1267


/Users/jamesyang/.pyenv/versions/3.12.2/envs/aipel/lib/python3.12/site-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [7]:
# 1. 파일 로드
data_path = f"{CFG['DATA_PATH']}/kochatgpt_1_SFT_augmented.jsonl"
dataset = load_dataset('json', data_files=data_path)

def preprocess_function(examples, prompt_template):
    sources = [prompt_template.format(prompt=p) for p in examples['prompt']]
    targets = [f"{c}{tokenizer.eos_token}" for c in examples['completion']]
    full_sentences = [s + t for s, t in zip(sources, targets)]
    
    # 1. 전체 문장 토크나이징 (Left Padding 적용)
    model_inputs = tokenizer(
        full_sentences, 
        max_length=CFG["MODEL_MAX_LEN"], 
        truncation=True, 
        padding="max_length"
    )
    
    labels = []
    for i in range(len(sources)):
        # 현재 행의 input_ids 가져오기
        input_ids = model_inputs["input_ids"][i]
        
        # 질문(source)과 전체 문장(full)의 토큰화된 실제 길이를 계산
        source_len = len(tokenizer(sources[i], truncation=True, add_special_tokens=False)["input_ids"])
        full_len = len(tokenizer(full_sentences[i], truncation=True, add_special_tokens=False)["input_ids"])
        
        # 학습해야 할 답변(target)의 실제 토큰 수 계산
        response_len = full_len - source_len
        
        # 레이블 초기화: 모두 -100으로 채움 (학습 제외)
        label = [-100] * len(input_ids)
        
        # 🚀 중요: Left Padding이므로 데이터는 항상 '맨 뒤'에 붙어 있습니다.
        # 따라서 맨 뒤에서부터 response_len 만큼만 실제 ID로 채웁니다.
        label[-response_len:] = input_ids[-response_len:]
        labels.append(label)
    
    model_inputs["labels"] = labels
    return model_inputs


tokenized_datasets = dataset.map(
    preprocess_function, 
    batched=True,
    fn_kwargs={"prompt_template": CFG["PROMPT_TEMPLATE"]}, 
    remove_columns=dataset['train'].column_names
)
# 4. 데이터 스플릿
split_dataset = tokenized_datasets['train'].train_test_split(test_size=0.1)
train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']
print(f"✅ 데이터 준비 및 스플릿 완료! (학습용 샘플: {len(train_dataset)})")

✅ 데이터 준비 및 스플릿 완료! (학습용 샘플: 10800)


In [ ]:
# [검증] 첫 번째 데이터를 디코딩해서 확인해보기
sample = train_dataset[0]
# 1. 원본 문장(input_ids) 확인
print("--- [1. 실제 입력 문장] ---")
print(tokenizer.decode(sample['input_ids'], skip_special_tokens=True)[:200] + "...")


# 2. 학습 대상(labels) 확인 (마스킹 확인)
print("\n--- [2. 모델이 학습하는 부분 (Labels)] ---")
# labels에서 -100이 아닌 부분만 추출해서 디코딩
labels = sample['labels']
actual_labels = [token if token != -100 else tokenizer.pad_token_id for token in labels]
decoded_labels = tokenizer.decode(actual_labels, skip_special_tokens=True)
print(f"가려진 부분(처음 10개): {labels[:10]}") # -100이 주르륵 나와야 함
print(f"실제 학습되는 텍스트: {decoded_labels.strip()[:150]}...")

--- [1. 실제 입력 문장] ---
### Instruction(명령어):
잠만 잤네

### Response(응답):어디서 자고 언제 자셨나요? 일어나서 하루도 잘 보내세요...

--- [2. 모델이 학습하는 부분 (Labels)] ---
가려진 부분(처음 10개): [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100]
실제 학습되는 텍스트: 어디서 자고 언제 자셨나요? 일어나서 하루도 잘 보내세요...
현재 사용 중인 장치: mps:0


In [ ]:
def run_sft_training(model, tokenizer, train_dataset, eval_dataset, cfg):
    """
    SFT(Supervised Fine-Tuning) 학습을 수행하고 모델을 저장하는 함수
    (Optimized for Mac M4 / 1.2B Model)
    """
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    # 2. 학습 파라미터 설정
    training_args = TrainingArguments(
        output_dir=cfg["TRAIN_OUTPUT_PATH"],
        per_device_train_batch_size=1,        
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,        # 실질적 배치 사이즈 8 (1 * 8)
        num_train_epochs=1,                   
        learning_rate=2e-4,                   
        # 🚀 Mac M4 가속 최적화 설정
        fp16=False,                           # MPS 이슈 방지
        bf16=True,                            # M4 칩 성능 극대화 (추천)
        logging_steps=20,                     
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=200,
        weight_decay=0.01,
        lr_scheduler_type="cosine",           
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",    # 가장 성적이 좋은(Loss 낮은) 시점 저장
        report_to="none",
        disable_tqdm=False,           # 👈 진행바를 강제로 활성화
        log_level="info",             # 👈 로그 수준을 높여서 상황을 더 자세                     
    )

    # 3. Trainer 초기화
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
    )

    # 4. 학습 시작 🚀
    print(f"🚀 [SFT Phase] {cfg['MODEL_NAME']} + LoRA 학습을 시작합니다...")
    trainer.train()

    # 5. 최적의 모델(Best Model) 및 토크나이저 저장
    trainer.save_model(cfg["MODEL_OUTPUT_PATH"])
    tokenizer.save_pretrained(cfg["MODEL_OUTPUT_PATH"])
    
    print(f"✅ 학습 완료! 최적 모델이 '{cfg['MODEL_OUTPUT_PATH']}'에 저장되었습니다.")
    return trainer # 👈 이 줄을 꼭 추가해 주세요!
   

# --- [함수 호출 예시] ---
if CFG["IS_SFT_TRAINING"] == True:
    print(f"현재 사용 중인 장치: {model.device}")
    trainer = run_sft_training(model, tokenizer, train_dataset, eval_dataset, CFG)


PyTorch: setting up devices
***** Running training *****
  Num examples = 10,800
  Num Epochs = 1
  Num update steps per epoch = 1,350
  Instantaneous batch size per device = 1
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 8
  Total optimization steps = 1,350
  Number of trainable parameters = 1,474,560


현재 사용 중인 장치: mps:0
🚀 [SFT Phase] skt/ko-gpt-trinity-1.2B-v0.5 + LoRA 학습을 시작합니다...


Step,Training Loss,Validation Loss
100,2.216999,2.128920
200,2.251865,2.079373
300,2.199605,2.055225
400,2.109027,2.040489
500,2.153045,2.024447
600,2.148133,2.015939
700,2.137295,2.006163
800,2.141697,1.998955
900,2.050648,1.994531
1000,2.102832,1.989989



***** Running Evaluation *****
  Num examples = 1200
  Batch size = 1

***** Running Evaluation *****
  Num examples = 1200
  Batch size = 1
Saving model checkpoint to kochatgpt-trinity-sft/checkpoint-200
loading configuration file config.json from cache at /Users/jamesyang/.cache/huggingface/hub/models--skt--ko-gpt-trinity-1.2B-v0.5/snapshots/2c9a93e78a61a52de058b66246f6acbd7b7c33b0/config.json
Model config GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 0,
  "embd_pdrop": 0.1,
  "eos_token_id": 8,
  "gradient_checkpointing": false,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 1920,
  "n_head": 16,
  "n_inner": 7680,
  "n_layer": 24,
  "n_positions": 1024,
  "pad_token_id": 8,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights

In [ ]:
# 수정 및 보완된 제안 코드
def save_merged_sft(): # save_mered_sft 오타 수정
    print("🚀 SFT 모델 병합(Merging) 시작...")
    
    # 1. 베이스 모델 로드
    base_model = AutoModelForCausalLM.from_pretrained(
        CFG["MODEL_NAME"], 
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    
    # 2. LoRA 어댑터 부착 및 병합
    model = PeftModel.from_pretrained(base_model, CFG["MODEL_OUTPUT_PATH"])
    merged_model = model.merge_and_unload()
    
    # 3. 토크나이저 준비
    tokenizer = PreTrainedTokenizerFast.from_pretrained(CFG["MODEL_OUTPUT_PATH"])
    
    # 4. 저장
    merged_model.save_pretrained(CFG["MERGED_SFT_PATH"])
    tokenizer.save_pretrained(CFG["MERGED_SFT_PATH"])
    print(f"✅ 병합된 모델이 '{CFG['MERGED_SFT_PATH']}'에 저장되었습니다.")

# 실행 조건
if CFG["IS_SFT_TRAINING"]:
    save_merged_sft()


In [ ]:
import matplotlib.pyplot as plt

def plot_learning_curves(trainer):
    # 학습 기록 가져오기
    history = trainer.state.log_history
    
    train_loss = []
    eval_loss = []
    train_steps = []
    eval_steps = []
    
    for log in history:
        if 'loss' in log:  # Train Loss 기록 (logging_steps 마다)
            train_loss.append(log['loss'])
            train_steps.append(log['step'])
        if 'eval_loss' in log:  # Validation Loss 기록 (eval_steps 마다)
            eval_loss.append(log['eval_loss'])
            eval_steps.append(log['step'])
            
    # 그래프 그리기
    plt.figure(figsize=(10, 5))
    plt.plot(train_steps, train_loss, label='Train Loss', color='#1f77b4', alpha=0.5)
    plt.plot(eval_steps, eval_loss, label='Eval Loss', color='#d62728', marker='o', linewidth=2)
    
    plt.title('SFT Training Learning Curves', fontsize=15)
    plt.xlabel('Steps', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.show()

if CFG["IS_SFT_TRAINING"] == True:
    plot_learning_curves(trainer)

In [ ]:
import gc
import torch

# 1. 변수 참조 제거
if 'model' in globals(): del model
if 'tokenizer' in globals(): del tokenizer
if 'trainer' in globals(): del trainer

# 2. 파이썬 가비지 컬렉션 강제 실행
gc.collect()
# 3. 🚀 핵심: Mac MPS 캐시 완전히 비우기
if torch.backends.mps.is_available():
    torch.mps.empty_cache()
    
print("✨ [Memory Clean] Mac M4의 메모리 청소를 완료했습니다!")

여기서 PEFT는 Parameter-Efficient Fine-Tuning의 약자로, **"적은 수의 파라미터만 사용해서 효율적으로 미세 조정한다"**는 뜻입니다.

왜 이걸 쓰는지, 어떤 역할을 하는지 3 줄 요약해 드릴게요.

- "전부를 가르치지 않습니다" (얼리기)
원래 Trinity 1.2B 같은 거대 모델을 처음부터 끝까지 다 가르치려면 메모리가 수십 GB가 필요하고 시간도 며칠씩 걸립니다. PeftModel은 원래 모델의 거대한 가중치들은 절대 변하지 않게 꽁꽁 얼려버리고(Freeze), 그 옆에 아주 작은 **'지식 칩(LoRA 어댑터)'**만 붙입니다.

- "지식 칩(Adapter)의 집" (껍데기)
PeftModel은 **[원본 거대 모델] + [우리가 새로 학습시킨 작은 지식 조각]**을 하나로 묶어주는 포장지 같은 역할을 합니다. 우리가 model.generate()라고 명령을 내리면, PeftModel이 알아서 원본 모델의 지식과 우리가 새로 가르친 지식을 잘 버무려서 정답을 내놓게 됩니다.

- "용량 다이어트" (실무적 이유)
전체 모델을 학습하면 3GB짜리 모델 파일을 새로 저장해야 하지만, PeftModel을 쓰면 우리가 학습한 LoRA 어댑터만 따로 저장할 수 있습니다. 그래서 결과물 파일 용량이 수십 MB 수준으로 확 줄어듭니다.

In [ ]:
# 모델 로드 및 LoRA 어댑터 결합
print("🚀 원본 모델(Base Model) 로드 중...")
base_model = AutoModelForCausalLM.from_pretrained(
    CFG["MODEL_NAME"],      # "skt/ko-gpt-trinity-1.2B-v0.5"
    torch_dtype=torch.float16,
    device_map="auto"
)

print("🚀 LoRA 어댑터 부착 중...")
model = PeftModel.from_pretrained(base_model, CFG["MODEL_OUTPUT_PATH"])
model.eval() # 추론 모드로 전환


# 토크나이저 로드 (학습 시 저장한 경로에서 가져옴)
tokenizer = PreTrainedTokenizerFast.from_pretrained(CFG["MODEL_OUTPUT_PATH"])

# 평가용 메트릭 로드
# 어원: Recall-Oriented Understudy for Gisting Evaluation
# 설명: 모델이 낸 답변이 실제 정답 데이터에 포함된 단어들을 얼마나 많이 '포함(Recall)'하고 있는지 측정합니다.
rouge = evaluate.load("rouge")

# 어원: BiLingual Evaluation Understudy
# 설명: 모델이 생성한 문장의 단어 조합이 정답 문장의 단어 조합과 얼마나 '일치(Precision)'하는지 측정합니다.
bleu = evaluate.load("sacrebleu")


def ask(prompt, prompt_template, max_new_tokens=128):
    # 매개변수로 받은 템플릿 사용
    input_text = prompt_template.format(prompt=prompt)
    # ### Instruction: 안녕 ... ### Response: 로 포장
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,  # 창의성 조절
            top_p=0.9,  # 답변의 필터링
            repetition_penalty=1.2, # 같은 말 반복 방지
            do_sample=True, # 매번 조금씩 다른 답변 생성
            eos_token_id=tokenizer.eos_token_id
        )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # 템플릿의 마지막 줄을 찾아 답변 부분만 분리
    separator = prompt_template.split("\n")[-1].strip()
    if separator in generated_text:
        answer = generated_text.split(separator)[-1].strip()
    else:
        answer = generated_text.strip()
        
    return answer


# 정성적 확인 (직접 질문 던지기)
print("\n--- [정성적 평가: 생성 결과 확인] ---")
test_prompts = [
    "서울역사박물관은 어떤 곳인가요?",
    "점심 메뉴로 매콤한 음식을 추천해줘.",
    "인공지능이란 무엇인지 짧게 설명해봐."
]
for p in test_prompts:
    # 💡 템플릿 인자를 명시적으로 전달합니다.
    response = ask(p, CFG["PROMPT_TEMPLATE"]) 
    print(f"Q: {p}\nA: {response}\n{'-'*30}")



# 정량적 평가 (샘플링하여 ROUGE 점수 확인)
print("\n--- [정량적 평가: ROUGE 스코어 확인 중] ---")

# 10개의 문장...
eval_subset = eval_dataset.select(range(min(10, len(eval_dataset))))
# refs 정답, preds AI의 답
preds, refs = [], []


for sample in eval_subset:
    # 텍스트 디코딩 및 프롬프트 추출
    full_text = tokenizer.decode(sample['input_ids'], skip_special_tokens=True)
    raw_prompt = full_text.split("### Instruction(명령어):")[1].split("### Response(응답):")[0].strip()
  
    
    preds.append(ask(raw_prompt, CFG["PROMPT_TEMPLATE"], max_new_tokens=64))
    
    actual_label = [t for t in sample['labels'] if t != -100]
    refs.append(tokenizer.decode(actual_label, skip_special_tokens=True))


# 메트릭 계산 및 출력
if preds and refs:
    # 1. ROUGE 점수 계산
    rouge_results = rouge.compute(predictions=preds, references=refs)
    
    # 2. BLEU 점수 계산 (sacrebleu는 reference를 리스트의 리스트 형태로 받습니다)
    # sacrebleu는 0~100점 사이로 점수를 줍니다.
    bleu_results = bleu.compute(predictions=preds, references=[[r] for r in refs])
    
    print("\n" + "="*50)
    print("📊 [최종 정량적 평가 결과]")
    print(f"✅ ROUGE-L Score: {rouge_results['rougeL']:.4f} (내용 유사도)")
    print(f"✅ BLEU Score   : {bleu_results['score']:.2f} (문장 유창성)")
    print("="*50)

🚀 원본 모델(Base Model) 로드 중...


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: skt/ko-gpt-trinity-1.2B-v0.5
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🚀 LoRA 어댑터 부착 중...

--- [정성적 평가: 생성 결과 확인] ---
Q: 서울역사박물관은 어떤 곳인가요?
A: '서울역사박물관입니다.
------------------------------
Q: 점심 메뉴로 매콤한 음식을 추천해줘.
A: '저는 인공지능 언어모델로써, 일반적으로 어떤 음식이 점심에 적합한지 알 수 없습니다. 하지만, 일반적으로는 매운맛이 나면서 부드러운 식감의 음식들이 좋습니다.
------------------------------
Q: 인공지능이란 무엇인지 짧게 설명해봐.
A: '인공 지능은 인간의 사고와 판단을 모방하는 컴퓨터 프로그램으로, 인공지능의 한 분야입니다.\n\n인간이 자연이나 사물에서 얻은 정보를 바탕으로 지식을 축적하고 이를 토대로 새로운 가치를 창출하여 보다 나은 삶을 영위할 수 있도록 돕습니다. 또한, 인간 고유의 능력 중 하나인 창의성과 소통 능력을 향상시키는 데에도 도움을 줍니다.\n\n그러나 이러한 인공적인 AI는 인간이 만들어 낸 것이 아니라 자연의 일부이며, 자연 속에서 인간과 상호작용하며 발전해왔기 때문에 인지와 학습, 추론 등의 능력은 자연적으로 형성되는 것입니다. 따라서 인공 지능 역시 인간의 기술과 노하우가 투입되어 만들어진 기술이라고 할 수 있습니다.
------------------------------

--- [정량적 평가: ROUGE 스코어 확인 중] ---

📊 [최종 정량적 평가 결과]
✅ ROUGE-L Score: 0.0286 (내용 유사도)
✅ BLEU Score   : 1.38 (문장 유창성)


# [Phase 2] Reward Model (RM) 학습 코드

In [12]:
# 1. 전략(Strategy) 객체 생성 및 컨텍스트 초기화
import torch
from chatgpt.models.gpt import GPTCritic

strategy = NaiveStrategy()

with strategy.model_init_context():
    # RM은 질문과 답변을 보고 '점수'를 매겨야 하므로 SFT 완료 모델(병합본)을 바탕으로 학습합니다.
    # 🚀 Mac M4 가속을 위해 bfloat16()으로 정밀도 설정
    rm_model = GPTCritic(
        pretrained="models/SFT_merged", 
        lora_rank=CFG["LORA_RANK"]
    ).to(CFG["DEVICE"]).bfloat16()

# 메모리 절약을 위한 그래디언트 체크포인팅 활성화
rm_model.model.gradient_checkpointing_enable()

print("✅ SFT 병합 모델로부터 RM 학습용 모델(BF16) 로드 완료!")

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: skt/ko-gpt-trinity-1.2B-v0.5
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import json
from datasets import load_dataset

rm_data_path = f"{CFG['KOCHATGPT_PATH']}/kochatgpt_2_RM.jsonl"

with open(rm_data_path, "r", encoding='utf-8-sig') as f:
    raw_rm_data = json.load(f)

# 🚀 데이터 증량 및 정제 함수 정의
def build_augmented_rm_data(raw_data):
    pairwise_data = []
    
    # 전략 1: 최고의 성적순과 최악의 성적순을 명확히 대조 (Ranking 차이 극대화)
    for item in raw_data:
        prompt = item['prompt']
        ranking = item['ranking']
        winner_idx = ranking.index(0)               # 1등 답변 (Chosen)
        worst_idx = ranking.index(max(ranking))     # 꼴찌 답변 (Rejected)
        
        pairwise_data.append({
            'prompt': prompt,
            'chosen': item[f'completion_{winner_idx}'],
            'rejected': item[f'completion_{worst_idx}']
        })

    # KorQuAD 2.0 스타일 고품질 데이터 추가 (데이터 증량)
    # 실제 정답 데이터를 chosen으로, 모델이 헛소리한 것을 rejected로 시뮬레이션
    try:
        print("💡 외부 데이터셋(KorQuAD 기반) 로드 중...")
        ext_dataset = load_dataset("squad_kor_v1", split="train[:500]") # 500개 추가
        for doc in ext_dataset:
            pairwise_data.append({
                'prompt': doc['question'],
                'chosen': doc['answers']['text'][0],
                'rejected': "질문에 대한 정확한 답을 찾을 수 없습니다." # 저품질 예시
            })
        print(f"✅ 데이터 증량 완료 (총 {len(pairwise_data)} 개 페어)")
    except Exception as e:
        print(f"⚠️ 외부 데이터 로드 실패({e}), 기존 데이터로만 진행합니다.")

    return pairwise_data


pairwise_data = build_augmented_rm_data(raw_rm_data)
train_rm_dataset = RewardDataset(pairwise_data, tokenizer, max_length=512)


💡 외부 데이터셋(KorQuAD 기반) 로드 중...
✅ 데이터 증량 완료 (총 10720 개 페어)


  0%|          | 0/10720 [00:00<?, ?it/s]

In [ ]:
# 학습 데이터 중 일부를 떼어서 모델이 채점을 얼마나 잘하고 있는지 확인합니다.
random.seed(42)
random.shuffle(pairwise_data)
eval_size = 100
train_data = pairwise_data[:-eval_size]
eval_data = pairwise_data[-eval_size:]

train_rm_dataset = RewardDataset(train_data, tokenizer, 512)
eval_rm_dataset = RewardDataset(eval_data, tokenizer, 512)

# 옵티마이저 설정 (LoRA 레이어만 학습할 수 있도록 파라미터 전달)
optimizer = torch.optim.AdamW(rm_model.parameters(), lr=5e-5, weight_decay=0.01)

# 3. RM Trainer 초기화
trainer = RewardModelTrainer(
    model=rm_model,
    strategy=NaiveStrategy(),
    optim=optimizer,
    train_dataset=train_rm_dataset,
    eval_dataset=eval_rm_dataset,
    batch_size=1,            # M4 메모리 안전성 확보
    max_epochs=1,            # 1에폭만으로도 기본적인 채점 감각을 익힙니다
)

# 4. RM 학습 시작 🚀
print("🚀 [RM Phase] Trinity 1.2B 채점 모델 학습을 시작합니다...")
trainer.fit(use_lora=CFG["LORA_RANK"]) 




  0%|          | 0/10620 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

🚀 [RM Phase] Trinity 1.2B 채점 모델 학습을 시작합니다...


Train epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Train step of epoch 0:   0%|          | 0/10620 [00:00<?, ?it/s]

AttributeError: 'TrinityRM' object has no attribute 'save_pretrained'

In [ ]:
#rm_model.save_pretrained(CFG["RM_MODEL_OUTPUT_PATH"])
rm_model.model.save_pretrained(CFG["RM_MODEL_OUTPUT_PATH"]) # .model 추가
tokenizer.save_pretrained(CFG["RM_MODEL_OUTPUT_PATH"])

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ 모델이 models/RM에 성공적으로 저장되었습니다.


('models/RM/tokenizer_config.json', 'models/RM/tokenizer.json')

In [15]:
def get_reward(prompt, answer):
    input_text = f"### Instruction(명령어):\n{prompt}\n\n### Response(응답):{answer}"
    input_ids = tokenizer.encode(input_text, return_tensors='pt').to(CFG["DEVICE"])
    # 모델의 forward pass를 통해 점수(스칼라 값)를 얻습니다.
    reward = rm_model(input_ids)
    return reward.item()


p = "대한민국의 수도는 어디인가요?"
a_good = "대한민국의 수도는 서울입니다."
a_bad = "수도는 물이 나오는 곳입니다."
print(f"Good Answer Score: {get_reward(p, a_good):.4f}")
print(f"Bad Answer Score: {get_reward(p, a_bad):.4f}")

Good Answer Score: -0.1484
Bad Answer Score: -0.1846


In [16]:
# [PPO 전처리] SFT 모델 가중치 병합 (Merge and Unload)
# 🚀 Actor의 바탕이 되는 SFT 모델은 LoRA 어댑터 형태(7.8MB)이므로 본체와 병합이 필수입니다.
# 🚀 RM(Reward Model)은 이미 Full Model(2.3GB) 상태이므로 병합 과정이 필요 없습니다.

def merge_sft_only(base_model_name, sft_adapter_path, save_path):
    if os.path.exists(save_path):
        print(f"✅ 이미 병합된 SFT 모델이 존재합니다: {save_path}")
        return
    
    print(f"🚀 SFT 모델 병합 중... ({sft_adapter_path} -> {save_path})")
    # 메모리 절약을 위해 CPU에서 병합 수행
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name, 
        torch_dtype=torch.bfloat16, 
        device_map="cpu"
    )
    peft_model = PeftModel.from_pretrained(base_model, sft_adapter_path)
    merged_model = peft_model.merge_and_unload()
    merged_model.save_pretrained(save_path)
    print(f"✅ SFT 병합 완료: {save_path}")

# SFT 모델만 병합 수행
merge_sft_only(CFG["MODEL_NAME"], CFG["MODEL_OUTPUT_PATH"], "models/SFT_merged")
print("✅ RM 모델은 기존 경로(models/RM)를 그대로 사용합니다.")

✅ 이미 병합된 SFT 모델이 존재합니다: models/SFT_merged
✅ RM 모델은 기존 경로(models/RM)를 그대로 사용합니다.


# [Phase 3] Proximal Policy Optimization (PPO) 학습 로직

In [17]:
# 1. 전략(Strategy) 객체 생성 및 컨텍스트 초기화
strategy = NaiveStrategy()

with strategy.model_init_context():
    # 🚀 Actor(SFT)는 병합된 모델 경로를 사용하여 지식 유실 방지
    # 🚀 Critic(RM)은 이미 Full Model인 기존 RM 폴더를 직접 사용
    # 🚀 Mac M4 가속을 위해 bfloat16()으로 정밀도 설정
    
    # (1) Actor: 직접 학습될 모델 (병합된 SFT 기반 + 신규 LoRA)
    actor = GPTActor(
        pretrained="models/SFT_merged", 
        lora_rank=CFG["LORA_RANK"]
    ).to(CFG["DEVICE"]).bfloat16()
    
    # (2) Critic: 보조 학습 모델 (기존 RM 기반 + 신규 LoRA)
    critic = GPTCritic(
        pretrained=CFG["RM_MODEL_OUTPUT_PATH"],  # "models/RM" 직접 사용
        lora_rank=CFG["LORA_RANK"]
    ).to(CFG["DEVICE"]).bfloat16()
    
    # (3) Initial Model (Reference): 기준점 모델 (학습 X)
    initial_model = GPTActor(
        pretrained="models/SFT_merged", 
        lora_rank=0
    ).to(CFG["DEVICE"]).bfloat16()
    for param in initial_model.parameters():
        param.requires_grad = False
    initial_model.model.eval()
    
    # (4) Reward Model: 고정 보상 모델 (학습 X)
    _temp_rm = GPTCritic(
        pretrained=CFG["RM_MODEL_OUTPUT_PATH"], # "models/RM" 직접 사용
        lora_rank=0
    ).to(CFG["DEVICE"]).bfloat16()
    
    reward_model = RewardModel(_temp_rm.model, _temp_rm.value_head).to(CFG["DEVICE"])
    for param in reward_model.parameters():
        param.requires_grad = False
    reward_model.eval()

# 그래디언트 체크포인팅 활성화 (메모리 절약)
actor.model.gradient_checkpointing_enable()
critic.model.gradient_checkpointing_enable()

print(f"✅ 병합된 SFT와 기존 RM으로부터 PPO용 Actor/Critic(BF16) 로드 완료!")

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

✅ 병합된 SFT와 기존 RM으로부터 PPO용 Actor/Critic(BF16) 로드 완료!


In [18]:
# PPO용 데이터 로드 (프롬프트만 사용)
ppo_data_path = f"{CFG['KOCHATGPT_PATH']}/kochatgpt_3_PPO.jsonl"
with open(ppo_data_path, "r", encoding='utf-8-sig') as f:
    raw_ppo_data = json.load(f)
    
# 프롬프트 리스트 추출
list_prompt = [tmp['prompt'] for tmp in raw_ppo_data]


tokenizer = PreTrainedTokenizerFast.from_pretrained(
    CFG["MODEL_OUTPUT_PATH"],
    bos_token='</s>', 
    eos_token='</s>', 
    unk_token='<unk>', 
    pad_token='<pad>', 
    mask_token='<mask>', 
    padding_side="right",
    model_max_length=512
)



# 토크나이징 함수 정의 (PPO 학습 중 실시간 토크나이징을 위해 필요)
def tokenize_fn(texts):
    batch = tokenizer(texts, return_tensors='pt', max_length=96, padding=True, truncation=True)
    return {k: v.to(CFG["DEVICE"]) for k, v in batch.items()}

In [19]:
actor_optim = torch.optim.AdamW(actor.parameters(), lr=1e-6)
critic_optim = torch.optim.AdamW(critic.parameters(), lr=1e-6)

(actor, actor_optim), (critic, critic_optim), reward_model, initial_model = strategy.prepare(
    (actor, actor_optim), (critic, critic_optim), reward_model, initial_model
)

In [20]:
# 2. PPO 트레이너 설정
# 이제 여기서 전달되는 actor, critic 등은 이미 .prepare()를 거쳐 
# 하드웨어 및 학습 환경에 최적화된 상태의 객체들입니다.
trainer = PPOTrainer(
    strategy=strategy,       # 새로 생성하기보다 변수화된 strategy 사용 권장
    actor=actor,             # 이미 prepare된 actor
    critic=critic,           # 이미 prepare된 critic
    reward_model=reward_model,
    initial_model=initial_model,
    actor_optim=actor_optim, # 이미 prepare된 optimizer
    critic_optim=critic_optim,# 이미 prepare된 optimizer
    tokenizer=tokenize_fn,
    max_epochs=1,
    train_batch_size=1,      
    experience_batch_size=4, 
    max_length=64,           
    do_sample=True,
    temperature=1.0,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id
)


# 2. PPO 학습 시작!
print("🚀 [PPO Phase] 최종 강화학습을 시작합니다...")
# 에피소드 수를 적절히 조절하여 전체 학습 시간을 관리하세요.
trainer.fit(list_prompt, num_episodes=10, max_timesteps=3, update_timesteps=3)

# 3. 최종 업그레이드된 챗봇 저장
# 이 모델이 마침내 SFT + RM + PPO를 모두 마친 'Custom ChatGPT'입니다.
actor.model.save_pretrained(CFG["PPO_MODEL_OUTPUT_PATH"])
tokenizer.save_pretrained(CFG["PPO_MODEL_OUTPUT_PATH"])
print(f"✅ 축하합니다! 최종 모델이 {CFG["PPO_MODEL_OUTPUT_PATH"]}에 저장되었습니다.")


🚀 [PPO Phase] 최종 강화학습을 시작합니다...


Episode [1/10]:   0%|          | 0/3 [00:00<?, ?it/s]

Train epoch [1/1]:   0%|          | 0/12 [00:00<?, ?it/s]

Episode [2/10]:   0%|          | 0/3 [00:00<?, ?it/s]

Train epoch [1/1]:   0%|          | 0/12 [00:00<?, ?it/s]

...

Train epoch [1/1]:   0%|          | 0/12 [00:00<?, ?it/s]

Episode [10/10]:   0%|          | 0/3 [00:00<?, ?it/s]

Train epoch [1/1]:   0%|          | 0/12 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ 축하합니다! 최종 모델이 models/PPO에 저장되었습니다.


In [29]:
# [최종 단계] 학습된 PPO 모델 테스트
from transformers import pipeline, logging as tf_logging
import torch
import warnings

# 1. 모든 불필요한 로그 및 경고 완벽 차단
warnings.filterwarnings("ignore")
tf_logging.set_verbosity_error()

# 2. 모델 로드
final_model_path = CFG["PPO_MODEL_OUTPUT_PATH"]

print(f"🚀 최종 PPO 모델 로드 완료! 테스트를 시작합니다.")
ppo_chat = pipeline(
    "text-generation", 
    model=final_model_path,
    tokenizer=final_model_path,
    device=CFG["DEVICE"],
    torch_dtype=torch.bfloat16
)

def ask_final(prompt):
    input_text = f"### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
    
    # max_length 경고를 피하기 위해 generation_config 대신 명시적 인자만 최소한으로 사용
    gen_args = {
        "max_new_tokens": 128,
        "do_sample": True,
        "top_p": 0.9,
        "temperature": 0.7,
        "repetition_penalty": 1.1,
        "eos_token_id": ppo_chat.tokenizer.eos_token_id,
        "pad_token_id": ppo_chat.tokenizer.pad_token_id,
    }
    
    output = ppo_chat(input_text, **gen_args)
    response = output[0]['generated_text'].split("### Response(응답):")[-1].strip()
    return response

# 3. 요청하신 프롬프트 리스트로 테스트 수행
list_prompt = [
    '불고기용 고기 한우에요?',
    '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
    '시카고 오헤어 국제공항은 어디에 있어',
    '오늘 미세먼지 어때?'
]

print("\n" + "="*50)
print("🤖 PPO 학습 완료 모델의 답변 확인")
print("="*50)

for p in list_prompt:
    print(f"\nQ: {p}")
    ans = ask_final(p)
    print(f"A: {ans}")
    print("-" * 30)

🚀 최종 PPO 모델 로드 완료! 테스트를 시작합니다.


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]


🤖 PPO 학습 완료 모델의 답변 확인

Q: 불고기용 고기 한우에요?
A: '저는 AI 어시스턴트이므로, 불고기용 고기에 대한 정보를 알 수 없습니다. 하지만, 일반적으로 쇠고기는 지방과 단백질이 풍부하고, 소의 종류에 따라 다양한 종류가 있으며, 또한 부위별로 맛이 다양하므로 참고하시기 바랍니다.
------------------------------

Q: 리처드 닉슨이 43대 부통령직을 수행한 년도는?
A: '1991년입니다.
------------------------------

Q: 시카고 오헤어 국제공항은 어디에 있어
A: '저는 시카고 오헤어 국제공항의 정보를 알 수 없습니다. 해당 공항에 대한 정보가 있으면 답변해드리겠습니다.
------------------------------

Q: 오늘 미세먼지 어때?
A: '저는 인공지능 어시스턴트이므로 미세먼지 농도나 시간 등을 알 수 없습니다. 하지만 일반적으로 미세먼지 농도는 보통 200~300g/m3 정도입니다. 따라서, 야외활동을 하거나 미세먼지가 심한 날에는 창문을 닫고 실내에서 생활하시는 것이 좋습니다.
------------------------------
